# 02 · Building grounded financial RAG
### BeliefLens · STAC AI Workshop · NYC · 21 October 2026

## Learning objectives
1. Enforce publication and availability times
2. Trace provenance
3. Detect missing evidence and conflicts
4. Measure retrieval dependence

All companies, numbers, labels and market observations in these notebooks are synthetic teaching examples. Offline outputs come from explicit deterministic rules, not an LLM. The same JavaScript case engine runs in the browser and through this Python client. Only a cell explicitly using `live=True` calls Together AI. No financial action is executed.

**Requirements:** Python 3.10+, Node.js 20+, Jupyter. No model weights or API key are needed for the default exercises. Run cells in order.

In [ ]:
from pathlib import Path
import sys, json, math, random, statistics
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_client.py").exists()), None)
if root is None:
    raise RuntimeError("Extract the complete workshop ZIP and open Jupyter inside its folder.")
if str(root) not in sys.path: sys.path.insert(0, str(root))
from lab_client import run, compact, save
print("Shared engine ready:", compact(run()))

## Time is part of evidence
The decision cutoff is 08:55 New York (12:55 UTC). The 09:05 amendment is relevant but unavailable. In a real system, ingestion availability can lag publication. Both must precede the cutoff. This lab uses explicit source selection, not an embedding search service.

In [ ]:
from datetime import datetime
baseline = run()
cutoff = datetime.fromisoformat("2026-10-20T12:55:00+00:00")
for source in baseline["sources"]:
    eligible = all(datetime.fromisoformat(source[k].replace("Z", "+00:00")) <= cutoff for k in ["published", "available"])
    print(source["id"], source["title"], "eligible:", eligible)
    assert eligible == source["eligible"]

## Inject faults and inspect the check
The future-leak switch intentionally bypasses the retrieval filter so you can test whether the downstream gate catches the fault.

In [ ]:
for fault in [{}, {"futureLeak": True}, {"dropCovenant": True}]:
    result = run(fault)
    print(fault, compact(result))
    for c in result["checks"][:2]: print(c["id"], c["reason"])
assert run({"futureLeak":True})["checks"][0]["status"] == "UNRESOLVED"
assert run({"dropCovenant":True})["checks"][1]["status"] == "UNRESOLVED"

## A simple retrieval ranking exercise
Lexical overlap is intentionally simple. It demonstrates that relevance and admissibility are different operations; a production retriever would also require coverage, access checks and retrieval evaluation.

In [ ]:
query = set("covenant ceiling amended".split())
ranked = sorted(baseline["sources"], key=lambda s: len(query & set(s["text"].lower().split())), reverse=True)
print("Before temporal filtering:", [x["id"] for x in ranked])
print("After temporal filtering:", [x["id"] for x in ranked if x["eligible"]])

## Learning check
A correct-looking answer cites E4. Can an approver repair it simply by signing? **No:** remove inadmissible context and regenerate/reassess. Approval cannot make future evidence historically available.

**Discussion:** Who defines required coverage? A credit expert defines the covenant requirements; data owners attest versions and availability; governance owners define escalation. A graph records these dependencies but does not invent the business policy.

## What you have demonstrated
- **Enforce publication and availability times:** Computed eligibility from both timestamps
- **Trace provenance:** Inspected source IDs, text and dates
- **Detect missing evidence and conflicts:** Removed a required covenant and injected future context
- **Measure retrieval dependence:** Observed changes in R1/R2 and the gate

**Exit ticket:** State one result, the evidence supporting it, and one limit that prevents you from claiming more. Save a run JSON or your numerical results as the experiment record.